# Lesson 5.1 — Preserving Critical Information Across Long Interactions

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain progressive-summarization risk and the 'lost in the middle' effect
- Maintain a persistent case-facts block outside summarized history
- Trim verbose tool outputs to only relevant fields before they accumulate
- Place key findings at the start/end and use explicit section headers

**You will produce:** A persistent case-facts block plus a tool-output trimmer and position-aware layout.

## Phase 5 Overview

Phase 5 covers **Domain 5 - Context Management & Reliability (15%)**. You will preserve critical facts across long context, design escalation/ambiguity handling, propagate errors across multi-agent systems, manage large-codebase context, build human-review/confidence calibration, and preserve provenance in multi-source synthesis.

By the end of this phase, you will have built:

- A persistent case-facts block + tool-output trimmer (Lesson 5.1)
- Escalation/ambiguity decision logic (Lesson 5.2)
- Structured error propagation across agents (Lesson 5.3)
- Scratchpads, /compact, and crash-recovery manifests (Lesson 5.4)
- Stratified sampling + field-level confidence routing (Lesson 5.5)
- Claim-source provenance and conflict/temporal handling (Lesson 5.6)

### Prerequisites

- Phases 1-4 complete
- Python 3 kernel with the `anthropic` SDK; optional `ANTHROPIC_API_KEY`
- Understanding of agentic loops (Phase 1) and multi-agent synthesis (Phases 1, 4)

## Concepts

Long interactions erode critical details. Two failure modes:

- **Progressive-summarization risk:** condensing numbers, percentages, dates, and customer-stated expectations into vague summaries loses exactly the facts you need.
- **"Lost in the middle":** models reliably use the **beginning and end** of long inputs but may omit findings from the **middle**.

Also, tool results accumulate and consume tokens disproportionately (40+ fields per order lookup when only 5 matter). Mitigations:
- Extract **transactional facts** (amounts, dates, order numbers, statuses) into a persistent **case-facts block** included in each prompt, **outside** summarized history.
- **Trim verbose tool outputs** to only relevant fields before they accumulate.
- Put **key findings at the beginning**, organize details with **explicit section headers**, and keep important items near the ends to mitigate position effects.
- Always pass **complete conversation history** in subsequent requests for coherence.

> **Reference:** [Anthropic - long context tips](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/long-context-tips)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A persistent case-facts block

Facts live verbatim in every prompt, immune to summarization drift.

In [ ]:
case_facts = {
    "customer_id": "CUST-8842",
    "order_number": "#12345",
    "refund_amount": 49.99,
    "promised_delivery": "2026-11-05",
    "status": "damaged_on_arrival",
}
def with_case_facts(history_summary):
    facts = "\n".join(f"{k}: {v}" for k, v in case_facts.items())
    return f"CASE FACTS (do not summarize):\n{facts}\n\nHISTORY:\n{history_summary}"

print(with_case_facts("[earlier turns summarized]")[:200])

## Code-Along: Step 2 — Trim verbose tool output

Keep only the return-relevant fields so context doesn't bloat.

In [ ]:
RELEVANT = {"order_id", "status", "total", "order_date", "items"}
def trim(order):
    return {k: v for k, v in order.items() if k in RELEVANT}

fat_order = {f"field_{i}": i for i in range(40)}
fat_order.update({"order_id": "#12345", "status": "shipped",
                  "total": 49.99, "order_date": "2026-11-01", "items": 2})
print('before:', len(fat_order), 'fields -> after:', len(trim(fat_order)))
print(trim(fat_order))

## Code-Along: Step 3 — Position-aware layout

Lead with a key-findings summary; use headers so middle content isn't lost.

In [ ]:
def layout(findings):
    head = "KEY FINDINGS (summary):\n" + "\n".join(f"- {f}" for f in findings[:3])
    body = "\n\n## DETAILS\n" + "\n".join(
        f"### Section {i+1}\n{f}" for i, f in enumerate(findings))
    return head + body

print(layout(["refund approved", "carrier damage", "goodwill credit", "notify ops"])[:260])

## Exercise

For a multi-issue session, build a `case_layer` that stores structured issue data (order IDs, amounts, statuses) separately from conversation. Then write an assertion that your trimmed order output preserves all five RELEVANT fields while dropping the rest - proving no transactional fact was summarized away.

In [ ]:
# TODO: build case_layer = {order_id: {amount, status}} for multiple issues
# TODO: assert RELEVANT <= set(trim(fat_order).keys())

## Key Takeaways

- Progressive summarization can drop numbers, dates, and customer expectations - keep a facts block.
- Lost-in-the-middle: put key info at the start/end with explicit section headers.
- Trim verbose tool outputs to only relevant fields before they accumulate.
- Pass complete conversation history for coherence; keep case facts outside summaries.

---

**Next:** [Lesson 5.2 — Escalation & Ambiguity Resolution](./lesson-5.2-escalation-ambiguity.ipynb)